<a href="https://colab.research.google.com/github/drtagkim/Lab/blob/master/Curses_Example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================================
#  Title : Curses Menu CLI Example
#  Author: Taekyung Kim, Professor, Big Data Analytics, Kyung Hee University
#  Description:
#     A reusable event-driven CLI menu system using Python curses.
#     Each menu item is defined as a dict { "MenuName": listener_function },
#     and the corresponding function is automatically triggered when selected.
# ================================================================

import curses

class CursesMenu:
    def __init__(self, stdscr, title="MS-DOS Emulator", menu_dict=None):
        """
        menu_dict: { '메뉴이름': 함수, ... }
        """
        self.stdscr = stdscr
        self.title = title
        self.menu_dict = menu_dict or {}
        self.menu_items = list(self.menu_dict.keys())
        self.current_row = 0
        self.highlight_color = 1

        curses.curs_set(0)
        curses.start_color()
        curses.init_pair(self.highlight_color, curses.COLOR_WHITE, curses.COLOR_BLUE)

    def print_menu(self):
        """화면에 메뉴를 출력"""
        self.stdscr.clear()
        self.stdscr.addstr(0, 2, self.title, curses.A_BOLD)
        self.stdscr.addstr(1, 0, "-" * 30)

        for idx, row in enumerate(self.menu_items):
            x, y = 2, 3 + idx
            padded_text = f"  {row:<18}"  # 20칸 고정 폭
            if idx == self.current_row:
                self.stdscr.attron(curses.color_pair(self.highlight_color))
                self.stdscr.addstr(y, x, padded_text)
                self.stdscr.attroff(curses.color_pair(self.highlight_color))
            else:
                self.stdscr.addstr(y, x, padded_text)
        self.stdscr.refresh()

    def run(self):
        """메뉴 메인 루프"""
        self.print_menu()
        while True:
            key = self.stdscr.getch()

            if key == curses.KEY_UP and self.current_row > 0:
                self.current_row -= 1
            elif key == curses.KEY_DOWN and self.current_row < len(self.menu_items) - 1:
                self.current_row += 1
            elif key in [curses.KEY_ENTER, 10, 13]:
                selected_item = self.menu_items[self.current_row]
                listener = self.menu_dict.get(selected_item)
                if callable(listener):
                    listener(self, selected_item)
                if selected_item.lower() in ['close', 'exit', 'quit']:
                    break
            self.print_menu()


# -----------------------------------------------------------
# 리스너 함수들

def file_action(menu_obj, name):
    menu_obj.stdscr.addstr(8, 2, f"[EVENT] '{name}' 선택됨 → 파일 메뉴 실행", curses.A_BOLD)
    menu_obj.stdscr.refresh()
    menu_obj.stdscr.getch()

def setting_action(menu_obj, name):
    menu_obj.stdscr.addstr(8, 2, f"[EVENT] '{name}' 선택됨 → 설정 메뉴 진입", curses.A_BOLD)
    menu_obj.stdscr.refresh()
    menu_obj.stdscr.getch()

def help_action(menu_obj, name):
    menu_obj.stdscr.addstr(8, 2, f"[EVENT] '{name}' 선택됨 → 도움말 표시", curses.A_BOLD)
    menu_obj.stdscr.refresh()
    menu_obj.stdscr.getch()

def close_action(menu_obj, name):
    menu_obj.stdscr.addstr(8, 2, "프로그램을 종료합니다.", curses.A_BOLD)
    menu_obj.stdscr.refresh()
    menu_obj.stdscr.getch()


# -----------------------------------------------------------
# 실행부

def main(stdscr):
    menu_dict = {
        'File': file_action,
        'Setting': setting_action,
        'Help': help_action,
        'Close': close_action
    }

    menu = CursesMenu(stdscr, title="MS-DOS Emulator", menu_dict=menu_dict)
    menu.run()


if __name__ == "__main__":
    curses.wrapper(main)